# Setup & Imports

In [1]:
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image
import os
import gc
import matplotlib.pyplot as plt

/Users/brageramberg/opt/miniconda3/envs/bannergen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
import os, psutil
print(f"Python using: {psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3):.2f} GB")

Python using: 0.16 GB


# Configuration


In [17]:

models = {
    "SD 1.5 + IP-Adapter": "runwayml/stable-diffusion-v1-5"
}

ip_adapter_path = "h94/IP-Adapter"
ip_weights = "ip-adapter_sd15.bin"

logo_path = "nike.png"
prompt = "A summer forest banner, bright lighting, detailed landscape"
negative_prompt = "ugly, blurry, deformed, low quality, text, watermark"
guidance_scale = 7.5
steps = 5
adapter_scale = 0.7

output_dir = "results/ip_adapter_comparison"
os.makedirs(output_dir, exist_ok=True)

device = "mps"  # safer default for Apple Silicon
dtype = torch.float16  # safer than float16 for MPS

print(f" Device: {device} | dtype: {dtype}")


 Device: mps | dtype: torch.float16


# Load and prepare the logo

In [18]:
logo = Image.open(logo_path).convert("RGB").resize((224, 224))  # IP-Adapter expects this shape

# Generate Image per Model

In [20]:

for name, model_id in models.items():
    print(f"\n#️⃣ Loading model: {name} - {model_id}")

    pipe = AutoPipelineForText2Image.from_pretrained(
        model_id,
        torch_dtype=dtype
    ).to(device)

    pipe.load_ip_adapter(ip_adapter_path, subfolder="models", weight_name=ip_weights)
    pipe.set_ip_adapter_scale(adapter_scale)

    pipe.vae.to(dtype)
    pipe.unet.to(dtype)

    generator = torch.Generator(device=device).manual_seed(42)

    print(f"🧠 Generating image with {name}...")
    result = pipe(
        prompt=prompt,
        ip_adapter_image=logo,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        generator=generator
    ).images[0]

    output_path = os.path.join(output_dir, f"{name.replace(' ', '_')}_with_logo.png")
    result.save(output_path)
    print(f"✅ Image saved to: {output_path}")



#️⃣ Loading model: SD 1.5 + IP-Adapter - runwayml/stable-diffusion-v1-5


Loading pipeline components...: 100%|██████████| 7/7 [00:06<00:00,  1.11it/s]


RuntimeError: MPS backend out of memory (MPS allocated: 18.16 GB, other allocations: 384.00 KB, max allowed: 18.13 GB). Tried to allocate 2.00 KB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

# Clean up

In [6]:

del pipe
gc.collect()
torch.mps.empty_cache()

In [ ]:

# 🖼️ Display results
fig, axs = plt.subplots(1, len(models), figsize=(7 * len(models), 6))

for i, name in enumerate(models):
    img_path = os.path.join(output_dir, f"{name.replace(' ', '_')}_with_logo.png")
    img = Image.open(img_path)
    axs[i].imshow(img)
    axs[i].set_title(name)
    axs[i].axis("off")

plt.tight_layout()
plt.show()

# Memory Cleanup if nececary

In [8]:
import os, psutil
process = psutil.Process(os.getpid())
print(f"🐍 Python is using {process.memory_info().rss / (1024 ** 3):.2f} GB of RAM")

🐍 Python is using 0.36 GB of RAM


In [7]:
gc.collect()
torch.mps.empty_cache()
print("🧽 Cleared memory.\n")

🧽 Cleared memory.

